# Session 10: SVM cont.

## Loss functons

In [ ]:
"""
Demo comparing common surrogate loss functions used for classification,
plotted as a function of the margin z = y * f(x) (the true label times
the model's raw score).

Losses plotted:
    0/1 loss            L(z) = 1{z < 0}
    Hinge loss (linear)  L(z) = max(0, 1 - z)              (SVM)
    Hinge loss (quadratic) L(z) = max(0, 1 - z)^2          (squared hinge)
    Log loss             L(z) = log(1 + exp(-z))           (logistic regression)
    Exponential loss     L(z) = exp(-z)                    (AdaBoost)

All of these are convex upper bounds on the 0/1 loss, which is why
they are used as differentiable surrogates for the (non-convex,
non-differentiable) classification objective at prediction time.
"""

import numpy as np
import matplotlib.pyplot as plt

Z_MIN, Z_MAX = -3, 3
N_POINTS = 500


def zero_one_loss(z):
    return (z < 0).astype(float)


def hinge_loss_linear(z):
    return np.maximum(0, 1 - z)


def hinge_loss_quadratic(z):
    return np.maximum(0, 1 - z) ** 2


def log_loss(z):
    # log(1 + exp(-z)), computed in a numerically stable way for large |z|
    return np.logaddexp(0, -z)


def exponential_loss(z):
    return np.exp(-z)

def huber_loss(z, DELTA):
    # Ensure input is a NumPy array for element-wise operations
    z = np.asarray(z)
    abs_z = np.abs(z)
    
    # Apply the piecewise formula element-wise
    return np.where(abs_z <= DELTA, 
                    0.5 * z**2, 
                    DELTA * (abs_z - 0.5 * DELTA))


def main():
    z = np.linspace(Z_MIN, Z_MAX, N_POINTS)

    plt.figure(figsize=(7.5, 6))

    # 0/1 loss drawn as a step function, since it is genuinely
    # discontinuous at z=0 (not a smooth curve like the others).
    plt.step(z, zero_one_loss(z), where="post", color="black",
              linewidth=1.5, label="0/1 loss")

    plt.plot(z, hinge_loss_linear(z), color="tab:blue", linewidth=2,
              label="hinge loss (linear, SVM)")
    plt.plot(z, hinge_loss_quadratic(z), color="tab:cyan", linewidth=2,
              linestyle="--", label="hinge loss (quadratic)")
    plt.plot(z, log_loss(z), color="tab:orange", linewidth=2,
              label="log loss (logistic)")
    plt.plot(z, exponential_loss(z), color="tab:red", linewidth=2,
              label="exponential loss (AdaBoost)")
    D = 0.5 # Hyperparameter DELTA for the Huber loss
    plt.plot(z, huber_loss(z,D), color="tab:green", linewidth=2,
              label="Huber loss with DELTA="+ str(D))

    plt.axvline(0, color="grey", linewidth=0.8, linestyle=":")
    plt.axhline(0, color="grey", linewidth=0.8, linestyle=":")

    plt.xlim(Z_MIN, Z_MAX)
    plt.ylim(0, 4)
    plt.xlabel("margin  $z = y \\cdot f(x)$")
    plt.ylabel("$J(z)$")
    plt.title("Surrogate loss functions and 0/1 loss")
    plt.legend(fontsize=9, loc="upper right")
    plt.tight_layout()
    plt.savefig("loss_functions_demo.png", dpi=150)
    print("Saved plot to loss_functions_demo.png")


if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_circles
from sklearn.svm import SVC
from sklearn.metrics.pairwise import rbf_kernel

def compute_effective_dimension(X, gamma, lam=1e-3):
    """
    Computes the scale-sensitive effective dimension of a dataset 
    mapped into an RBF kernel space.
    """
    # 1. Compute the n x n Kernel (Gram) Matrix
    K = rbf_kernel(X, gamma=gamma)
    
    # 2. Get the eigenvalues of the kernel matrix
    eigenvalues = np.linalg.eigvalsh(K)
    
    # 3. Clip any slightly negative eigenvalues due to numerical precision
    eigenvalues = np.maximum(eigenvalues, 0)
    
    # 4. Apply the effective dimension formula
    eff_dim = np.sum(eigenvalues / (eigenvalues + lam))
    
    return float(eff_dim)


# 1. Generate concentric circles dataset (linearly inseparable in 2D)
X, y = make_circles(n_samples=200, factor=0.5, noise=0.1, random_state=42)
GAMMA = 1.0

# 2. Fit a Support Vector Machine using the RBF (Gaussian) Kernel
# Gamma controls the radius of influence for individual support vectors
clf = SVC(kernel='rbf', gamma=GAMMA)
clf.fit(X, y)

# 3. Create a dense grid to map out the decision boundary boundaries
x_min, x_max = X[:, 0].min() - 0.3, X[:, 0].max() + 0.3
y_min, y_max = X[:, 1].min() - 0.3, X[:, 1].max() + 0.3
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 200), np.linspace(y_min, y_max, 200))

# 4. Evaluate the distance from the decision boundary for every point on the grid
Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
Z = Z.reshape(xx.shape)

# 5. Calculate the effective dimensions of the hilbert space in which the data is mapped
EFF_DIM = compute_effective_dimension(X, GAMMA, lam=1/C)

# 5. Plot the results
plt.figure(figsize=(8, 6))

# Plot the decision boundary surface (the contour line where distance == 0)
plt.contourf(xx, yy, Z, levels=[-np.inf, 0, np.inf], colors=['#abc9ea', '#efb3b3'], alpha=0.4)
boundary = plt.contour(xx, yy, Z, levels=[0], linewidths=2, colors='darkred')

# Plot individual data clusters
plt.scatter(X[y == 0, 0], X[y == 0, 1], color='blue', edgecolors='k', label='Class A (Outer)')
plt.scatter(X[y == 1, 0], X[y == 1, 1], color='orange', edgecolors='k', label='Class B (Inner)')

plt.clabel(boundary, fmt={0: 'wᵀΦ(x) + b = 0'}, inline=True, fontsize=10)
plt.title("Complex Decision Boundary with RBF Kernel, γ="+str(GAMMA))
plt.xlabel("x 1")
plt.ylabel("x 2")
plt.legend(loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("kernel_trick.png", dpi=150)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.metrics.pairwise import rbf_kernel


def generate_circles(n_points, noise=0.5):
    """Generate concentric circles dataset (linearly inseparable in 2D)"""
    X, y = make_circles(n_samples=n_points, factor=0.5, noise=noise, random_state=42)
    GAMMA = 1.0

    return X, y

def generate_spirals(n_points, noise=0.5):
    """Generates a twin spiral dataset."""
    n = np.sqrt(np.random.rand(n_points, 1)) * 780 * (2 * np.pi) / 360
    d1x = -np.cos(n) * n + np.random.randn(n_points, 1) * noise
    d1y = np.sin(n) * n + np.random.randn(n_points, 1) * noise
    
    # Spiral 1 (Class 0)
    data_1 = np.hstack((d1x, d1y))
    labels_1 = np.zeros(n_points)
    
    # Spiral 2 (Class 1) - rotated by pi radians
    data_2 = np.hstack((-d1x, -d1y))
    labels_2 = np.ones(n_points)
    
    X = np.vstack((data_1, data_2))
    y = np.hstack((labels_1, labels_2))
    return X, y

def compute_effective_dimension(X, gamma, lam=1e-3):
    """
    Computes the scale-sensitive effective dimension of a dataset 
    mapped into an RBF kernel space.
    """
    # 1. Compute the n x n Kernel (Gram) Matrix
    K = rbf_kernel(X, gamma=gamma)
    
    # 2. Get the eigenvalues of the kernel matrix
    eigenvalues = np.linalg.eigvalsh(K)
    
    # 3. Clip any slightly negative eigenvalues due to numerical precision
    eigenvalues = np.maximum(eigenvalues, 0)
    
    # 4. Apply the effective dimension formula
    eff_dim = np.sum(eigenvalues / (eigenvalues + lam))
    
    return float(eff_dim)

# 1. Generate 400 total data points forming twin spirals
X, y = generate_spirals(n_points=200, noise=0.2)


def main():
    
    # 2. Train SVM with RBF Kernel 
    # Higher gamma allows the boundary to curve more tightly around complex shapes
    GAMMA =0.5
    C=10.0
    clf = SVC(kernel='rbf', gamma=GAMMA, C=C)
    clf.fit(X, y)

    # 3. Create a dense grid to map the decision boundary surface
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

    # 4. Predict the distance metric across the grid
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # 5. Calculate the effective dimensions of the hilbert space in which the data is mapped
    EFF_DIM = compute_effective_dimension(X, GAMMA, lam=1/C)

    # 6. Plot the decision landscape and original data points
    plt.figure(figsize=(8, 8))
    plt.contourf(xx, yy, Z, levels=[-np.inf, 0, np.inf], colors=['#abc9ea', '#efb3b3'], alpha=0.3)
    plt.contour(xx, yy, Z, levels=[0], colors='darkred', linewidths=2)

    plt.scatter(X[y == 0, 0], X[y == 0, 1], color='blue', edgecolors='k', s=25, label='Class A')
    plt.scatter(X[y == 1, 0], X[y == 1, 1], color='orange', edgecolors='k', s=25, label='Class B')

    plt.title("Complex Decision Boundary with RBF Kernel, γ="+str(GAMMA) +"\nEffective dimensions of the hilbert space: " + f"{EFF_DIM:.2f}")
    plt.xlabel("Feature 1")
    plt.ylabel("Feature 2")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig("kernel_trick2.png", dpi=150)

if __name__ == "__main__":
    main()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.svm import SVC
from sklearn.datasets import make_circles
from sklearn.metrics.pairwise import rbf_kernel

GAMMA = 0.5
C = 10.0


def generate_circles(n_points, noise=0.1):
    """Generate concentric circles dataset (linearly inseparable in 2D)"""
    X, y = make_circles(n_samples=n_points, factor=0.5, noise=noise, random_state=42)
    return X, y


def generate_spirals(n_points, noise=0.5):
    """Generates a twin spiral dataset."""
    n = np.sqrt(np.random.rand(n_points, 1)) * 780 * (2 * np.pi) / 360
    d1x = -np.cos(n) * n + np.random.randn(n_points, 1) * noise
    d1y = np.sin(n) * n + np.random.randn(n_points, 1) * noise

    # Spiral 1 (Class 0)
    data_1 = np.hstack((d1x, d1y))
    labels_1 = np.zeros(n_points)

    # Spiral 2 (Class 1) - rotated by pi radians
    data_2 = np.hstack((-d1x, -d1y))
    labels_2 = np.ones(n_points)

    X = np.vstack((data_1, data_2))
    y = np.hstack((labels_1, labels_2))
    return X, y


def compute_effective_dimension(X, gamma, lam=1e-3):
    """
    Computes the scale-sensitive effective dimension of a dataset
    mapped into an RBF kernel space.
    """
    # 1. Compute the n x n Kernel (Gram) Matrix
    K = rbf_kernel(X, gamma=gamma)

    # 2. Get the eigenvalues of the kernel matrix
    eigenvalues = np.linalg.eigvalsh(K)

    # 3. Clip any slightly negative eigenvalues due to numerical precision
    eigenvalues = np.maximum(eigenvalues, 0)

    # 4. Apply the effective dimension formula
    eff_dim = np.sum(eigenvalues / (eigenvalues + lam))

    return float(eff_dim)


def fit_and_plot(ax, X, y, title, gamma=GAMMA, c=C):
    """Train an RBF-kernel SVM on (X, y) and draw its decision boundary,
    data points, and effective kernel-space dimension onto `ax`."""

    # 1. Train SVM with RBF kernel. Higher gamma allows the boundary to
    # curve more tightly around complex shapes.
    clf = SVC(kernel="rbf", gamma=gamma, C=c)
    clf.fit(X, y)

    # 2. Create a dense grid to map the decision boundary surface
    x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))

    # 3. Predict the distance metric across the grid
    Z = clf.decision_function(np.c_[xx.ravel(), yy.ravel()])
    Z = Z.reshape(xx.shape)

    # 4. Calculate the effective dimension of the Hilbert space the
    # data is mapped into
    eff_dim = compute_effective_dimension(X, gamma, lam=1 / c)

    # 5. Plot the decision landscape and original data points
    ax.contourf(xx, yy, Z, levels=[-np.inf, 0, np.inf], colors=["#abc9ea", "#efb3b3"], alpha=0.3)
    ax.contour(xx, yy, Z, levels=[0], colors="darkred", linewidths=2)

    ax.scatter(X[y == 0, 0], X[y == 0, 1], color="blue", edgecolors="k", s=25, label="Class A")
    ax.scatter(X[y == 1, 0], X[y == 1, 1], color="orange", edgecolors="k", s=25, label="Class B")

    ax.set_title(f"{title}\nRBF \u03b3={gamma}, C={c}  |  effective dim={eff_dim:.2f}")
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")
    ax.legend()
    ax.grid(True, alpha=0.3)


def main():
    # 1. Generate both datasets: 400 total points forming concentric
    # circles, and 400 total points forming twin spirals - both
    # linearly inseparable in the original 2D feature space.
    X_circles, y_circles = generate_circles(n_points=200, noise=0.1)
    X_spirals, y_spirals = generate_spirals(n_points=200, noise=0.2)

    # 2. Train an RBF-kernel SVM on each dataset and plot both decision
    # boundaries side by side in one figure.
    fig, (ax_circles, ax_spirals) = plt.subplots(1, 2, figsize=(15, 7.5))

    fit_and_plot(ax_circles, X_circles, y_circles, "Concentric Circles")
    fit_and_plot(ax_spirals, X_spirals, y_spirals, "Twin Spirals")

    fig.suptitle("Complex Decision Boundaries with an RBF Kernel", fontsize=14)
    fig.tight_layout()
    fig.savefig("kernel_trick_comparison.png", dpi=150)
    print("Saved plot to kernel_trick_comparison.png")


if __name__ == "__main__":
    main()